# Entity Resolution — Dataset Cleaning & Preprocessing

Cleans Source 1, Source 2, and Source 3 for later ML matching. No ML training, blocking, or pair generation is performed here.

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata


In [ ]:
# Load datasets
df1 = pd.read_csv("train_source1.tsv", sep="\t", encoding="utf-8")
df2 = pd.read_csv("train_source2.tsv", sep="\t", encoding="utf-8")
df3 = pd.read_csv("train_source3.tsv", sep="\t", encoding="utf-8")
df4 = pd.read_csv("train_ground_truth.tsv", sep="\t", encoding="utf-8")

print("Source 1:", df1.shape)
print("Source 2:", df2.shape)
print("Source 3:", df3.shape)
print("Ground Truth:", df4.shape)


Source 1: (11054, 4)
Source 2: (10860, 4)
Source 3: (10963, 4)
Ground Truth: (18234, 2)


In [ ]:
# Preserve original data
df1_raw = df1.copy()
df2_raw = df2.copy()
df3_raw = df3.copy()
df4_raw = df4.copy()

# Clean column names
for df in [df1, df2, df3, df4]:
    df.columns = df.columns.str.strip()


In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = text.replace("&", " and ")

    cleaned = []
    for char in text:
        category = unicodedata.category(char)
        if (category.startswith("L") or category.startswith("N") or
            category.startswith("M") or category.startswith("Z")):
            cleaned.append(char)
        else:
            cleaned.append(" ")

    return re.sub(r"\s+", " ", "".join(cleaned)).strip()

for df in [df1, df2, df3]:
    df["business_name_clean"] = df["business_name"].apply(normalize_text)
    df["business_address_clean"] = df["business_address"].apply(normalize_text)
    df["country_clean"] = df["country"].apply(normalize_text)


In [ ]:
LEGAL_SUFFIX_PATTERNS = [
    r"\bprivate limited\b", r"\bpvt limited\b", r"\bpvt ltd\b",
    r"\bprivate ltd\b", r"\blimited\b", r"\bltd\b",
    r"\bllp\b", r"\bllc\b", r"\bincorporated\b", r"\binc\b",
    r"\bcorporation\b", r"\bcorp\b", r"\bcompany\b", r"\bco\b",
    r"\blp\b", r"\bpc\b"
]

def clean_name_core(text):
    if not text:
        return ""
    for pattern in LEGAL_SUFFIX_PATTERNS:
        text = re.sub(pattern, " ", text)
    return re.sub(r"\s+", " ", text).strip()

for df in [df1, df2, df3]:
    df["name_core"] = df["business_name_clean"].apply(clean_name_core)


In [ ]:
ADDRESS_ABBREVIATIONS = {
    r"\bst\b": "street", r"\brd\b": "road", r"\bave\b": "avenue",
    r"\bav\b": "avenue", r"\bblvd\b": "boulevard", r"\bln\b": "lane",
    r"\bdr\b": "drive", r"\bapt\b": "apartment", r"\bste\b": "suite",
    r"\bhwy\b": "highway"
}

def clean_address(text):
    if not text:
        return ""
    for pattern, replacement in ADDRESS_ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

def extract_numbers(text):
    return " ".join(re.findall(r"\d+", text)) if text else ""

def sorted_tokens(text):
    return " ".join(sorted(text.split())) if text else ""

for df in [df1, df2, df3]:
    df["address_clean"] = df["business_address_clean"].apply(clean_address)
    df["address_numbers"] = df["address_clean"].apply(extract_numbers)
    df["name_tokens"] = df["business_name_clean"].apply(sorted_tokens)
    df["address_tokens"] = df["address_clean"].apply(sorted_tokens)


In [ ]:
for df in [df1, df2, df3]:
    df["name_length"] = df["business_name_clean"].str.len()
    df["address_length"] = df["address_clean"].str.len()
    df["name_word_count"] = df["business_name_clean"].str.split().str.len()
    df["address_word_count"] = df["address_clean"].str.split().str.len()
    df["name_first_char"] = df["name_core"].str[:1]

    df["name_missing"] = (df["business_name_clean"] == "").astype(int)
    df["address_missing"] = (df["address_clean"] == "").astype(int)
    df["country_missing"] = (df["country_clean"] == "").astype(int)


In [ ]:
# Remove only completely identical rows.
# Do NOT remove records merely because business names are duplicated.
df1 = df1.drop_duplicates().reset_index(drop=True)
df2 = df2.drop_duplicates().reset_index(drop=True)
df3 = df3.drop_duplicates().reset_index(drop=True)

text_columns = [
    "business_name_clean", "business_address_clean", "country_clean",
    "name_core", "address_clean", "address_numbers",
    "name_tokens", "address_tokens", "name_first_char"
]

numeric_columns = [
    "name_length", "address_length", "name_word_count",
    "address_word_count", "name_missing", "address_missing",
    "country_missing"
]

for df in [df1, df2, df3]:
    for col in text_columns:
        df[col] = df[col].fillna("")
    for col in numeric_columns:
        df[col] = df[col].fillna(0)


In [ ]:
# Final quality check
for name, df in [("Source 1", df1), ("Source 2", df2), ("Source 3", df3)]:
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

print("\nCleaned sample:")
print(df1[[
    "entity_id", "business_name", "business_name_clean", "name_core",
    "business_address", "address_clean", "address_numbers", "country_clean"
]].head(10).to_string(index=False))



Source 1
Shape: (11054, 20)
Missing values: 1
Duplicate rows: 0

Source 2
Shape: (10860, 20)
Missing values: 368
Duplicate rows: 0

Source 3
Shape: (10963, 20)
Missing values: 389
Duplicate rows: 0

Cleaned sample:
   entity_id                            business_name                      business_name_clean                  name_core                                                                                                          business_address                                                                                                     address_clean address_numbers country_clean
S1-925783039                      Orelee's Barbershop                      orelee s barbershop        orelee s barbershop                                                                                    1795 Westchester Drive, High Point, NC                                                                              1795 westchester drive high point nc            1795            us
S1-77388

In [ ]:
# Save cleaned datasets
df1.to_csv("source1_clean.csv", index=False, encoding="utf-8")
df2.to_csv("source2_clean.csv", index=False, encoding="utf-8")
df3.to_csv("source3_clean.csv", index=False, encoding="utf-8")
df4.to_csv("ground_truth_clean.csv", index=False, encoding="utf-8")

print("All cleaned datasets saved successfully.")


All cleaned datasets saved successfully.


In [ ]:
df2["source"] = "S2"
df3["source"] = "S3"

matches = pd.concat(
    [df2, df3],
    ignore_index=True
)

print("Combined S2 + S3:", matches.shape)
print(matches["source"].value_counts())

Combined S2 + S3: (21823, 21)
source
S3    10963
S2    10860
Name: count, dtype: int64


In [ ]:
positive_pairs = []

for _, row in df4.iterrows():

    s1_id = row["source1_entity_id"]

    if pd.isna(row["matched_entity_ids"]):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id:
            positive_pairs.append(
                (s1_id, match_id, 1)
            )

positive_pairs = pd.DataFrame(
    positive_pairs,
    columns=[
        "source1_entity_id",
        "matched_entity_id",
        "label"
    ]
)

# Remove accidental duplicate pairs
positive_pairs = positive_pairs.drop_duplicates()

print("Positive pairs:", len(positive_pairs))
print(positive_pairs.head())

Positive pairs: 63045
  source1_entity_id matched_entity_id  label
0         S1-965667      S2-681193310      1
1         S1-965667      S2-743505751      1
2         S1-965667      S3-775321672      1
3         S1-965667       S3-11291185      1
4         S1-965667      S3-860443364      1


In [ ]:
# ============================================================
# 17. CREATE BALANCED NEGATIVE PAIRS
# ============================================================

rng = np.random.default_rng(42)

positive_set = set(
    zip(
        positive_pairs["source1_entity_id"],
        positive_pairs["matched_entity_id"]
    )
)

all_match_ids = matches["entity_id"].dropna().unique()

negative_pairs = []

for s1_id in positive_pairs["source1_entity_id"]:

    while True:

        candidate = rng.choice(all_match_ids)

        # Make sure this is NOT a genuine match
        if (s1_id, candidate) not in positive_set:
            negative_pairs.append(
                (s1_id, candidate, 0)
            )
            break

negative_pairs = pd.DataFrame(
    negative_pairs,
    columns=[
        "source1_entity_id",
        "matched_entity_id",
        "label"
    ]
)

negative_pairs = negative_pairs.drop_duplicates(
    subset=["source1_entity_id", "matched_entity_id"]
).reset_index(drop=True)

print("Positive pairs:", len(positive_pairs))
print("Negative pairs:", len(negative_pairs))

Positive pairs: 63045
Negative pairs: 63039


In [ ]:
# ============================================================
# 18. CREATE SIMILARITY FEATURES
# ============================================================

def calculate_features(row):

    return pd.Series({

        "name_ratio": ratio(
            str(row["name_s1"]),
            str(row["name_match"])
        ) / 100,

        "name_token_set_ratio": token_set_ratio(
            str(row["name_s1"]),
            str(row["name_match"])
        ) / 100,

        "name_token_sort_ratio": token_sort_ratio(
            str(row["name_s1"]),
            str(row["name_match"])
        ) / 100,

        "core_name_ratio": ratio(
            str(row["core_name_s1"]),
            str(row["core_name_match"])
        ) / 100,

        "address_ratio": ratio(
            str(row["address_s1"]),
            str(row["address_match"])
        ) / 100,

        "address_token_set_ratio": token_set_ratio(
            str(row["address_s1"]),
            str(row["address_match"])
        ) / 100,

        "address_number_match": int(
            row["numbers_s1"] != "" and
            row["numbers_s1"] == row["numbers_match"]
        ),

        "country_match": int(
            row["country_s1"] != "" and
            row["country_s1"] == row["country_match"]
        ),

        "name_exact_match": int(
            row["name_s1"] == row["name_match"]
        ),

        "core_name_exact_match": int(
            row["core_name_s1"] == row["core_name_match"]
        ),

        "address_exact_match": int(
            row["address_s1"] == row["address_match"]
        )
    })


# Actually calculate the features
features = training_pairs.apply(
    calculate_features,
    axis=1
)

print("Features created:", features.shape)
print(features.head())

Features created: (126084, 11)
   name_ratio  name_token_set_ratio  name_token_sort_ratio  core_name_ratio  \
0    0.105263              0.105263               0.105263         0.105263   
1    0.058824              0.058824               0.058824         0.111111   
2    0.090909              0.090909               0.090909         0.142857   
3    1.000000              1.000000               1.000000         1.000000   
4    0.285714              0.285714               0.285714         0.285714   

   address_ratio  address_token_set_ratio  address_number_match  \
0       0.107143                 0.071429                   0.0   
1       0.125000                 0.125000                   0.0   
2       0.090909                 0.107143                   0.0   
3       1.000000                 1.000000                   0.0   
4       0.080000                 0.080000                   0.0   

   country_match  name_exact_match  core_name_exact_match  address_exact_match  
0         

In [ ]:
# ============================================================
# 19. CREATE FINAL ML TRAINING DATASET
# ============================================================

training_data = pd.concat(
    [
        training_pairs[
            [
                "source1_entity_id",
                "matched_entity_id",
                "label"
            ]
        ].reset_index(drop=True),

        features.reset_index(drop=True)
    ],
    axis=1
)

print("Training data shape:", training_data.shape)
print(training_data.head())

Training data shape: (126084, 14)
  source1_entity_id matched_entity_id  label  name_ratio  \
0      S1-941414043      S3-840721667      0    0.105263   
1      S1-594304013      S3-882605073      0    0.058824   
2      S1-117085755      S3-719204259      0    0.090909   
3      S1-870288035      S2-524962010      1    1.000000   
4      S1-630883950      S2-171255805      0    0.285714   

   name_token_set_ratio  name_token_sort_ratio  core_name_ratio  \
0              0.105263               0.105263         0.105263   
1              0.058824               0.058824         0.111111   
2              0.090909               0.090909         0.142857   
3              1.000000               1.000000         1.000000   
4              0.285714               0.285714         0.285714   

   address_ratio  address_token_set_ratio  address_number_match  \
0       0.107143                 0.071429                   0.0   
1       0.125000                 0.125000                   0.0   
2

In [ ]:
# ============================================================
# 20. VALIDATE FINAL ML DATASET
# ============================================================

print("Shape:")
print(training_data.shape)

print("\nClass distribution:")
print(training_data["label"].value_counts())

print("\nTotal missing values:")
print(training_data.isnull().sum().sum())

print("\nFeature statistics:")
print(
    training_data[
        [
            "name_ratio",
            "name_token_set_ratio",
            "name_token_sort_ratio",
            "core_name_ratio",
            "address_ratio",
            "address_token_set_ratio",
            "address_number_match",
            "country_match"
        ]
    ].describe()
)

Shape:
(126084, 14)

Class distribution:
label
1    63045
0    63039
Name: count, dtype: int64

Total missing values:
0

Feature statistics:
          name_ratio  name_token_set_ratio  name_token_sort_ratio  \
count  126084.000000         126084.000000          126084.000000   
mean        0.566404              0.566347               0.565918   
std         0.434244              0.434267               0.434653   
min         0.000000              0.000000               0.000000   
25%         0.137931              0.137931               0.137931   
50%         0.375000              0.375000               0.367007   
75%         1.000000              1.000000               1.000000   
max         1.000000              1.000000               1.000000   

       core_name_ratio  address_ratio  address_token_set_ratio  \
count    126084.000000  126084.000000            126084.000000   
mean          0.573109       0.551361                 0.551732   
std           0.428546       0.446819  

In [ ]:
# ============================================================
# 21. SAVE FINAL ML-READY DATASET
# ============================================================

training_data.to_csv(
    "training_data.csv",
    index=False,
    encoding="utf-8"
)

print("Final training_data.csv saved successfully!")

Final training_data.csv saved successfully!


In [ ]:
print(training_data.shape)
print(training_data["label"].value_counts())
print(training_data.isnull().sum().sum())

(126084, 14)
label
1    63045
0    63039
Name: count, dtype: int64
0


In [67]:
from google.colab import files
files.download('training_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df12 = pd.read_csv('training_data.csv')
display(df12.head(20))

,source1_entity_id,matched_entity_id,label,name_ratio,name_token_set_ratio,name_token_sort_ratio,core_name_ratio,address_ratio,address_token_set_ratio,address_number_match,country_match,name_exact_match,core_name_exact_match,address_exact_match
0,S1-941414043,S3-840721667,0,0.105263,0.105263,0.105263,0.105263,0.107143,0.071429,0.0,0.0,0.0,0.0,0.0
1,S1-594304013,S3-882605073,0,0.058824,0.058824,0.058824,0.111111,0.125000,0.125000,0.0,0.0,0.0,0.0,0.0
2,S1-117085755,S3-719204259,0,0.090909,0.090909,0.090909,0.142857,0.090909,0.107143,0.0,0.0,0.0,0.0,0.0
3,S1-870288035,S2-524962010,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0
4,S1-630883950,S2-171255805,0,0.285714,0.285714,0.285714,0.285714,0.080000,0.080000,0.0,0.0,0.0,0.0,0.0
5,S1-136452969,S2-783562662,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0
6,S1-247776121,S2-651967616,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0
7,S1-846115345,S3-730814053,0,0.214286,0.214286,0.214286,0.214286,0.105263,0.105263,0.0,0.0,0.0,0.0,0.0
8,S1-42918236,S2-174609481,0,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.0,0.0,0.0,0.0,0.0
9,S1-697619977,S3-805146789,0,0.100000,0.100000,0.100000,0.125000,0.089552,0.059701,0.0,0.0,0.0,0.0,0.0
